<div align="center">

# NB5 · Attack 2: Delaying ICU Sepsis Alerts

**Explainable Artificial Intelligence in Medical Systems: From Black Box Models to Reliable Clinical Decision Support**<br>
IEEE CARS 2026 Workshop · Grand Forks, ND, USA · 28 October 2026 · Session 11:00 to 11:30

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/utkukose/xai-cars2026-NB-workshop/blob/main/notebooks/NB5_Attack_ICU_Sepsis.ipynb)

Prof. Dr. Utku Köse · Süleyman Demirel University · University of North Dakota · Universidad Panamericana · Vel Tech University<br>
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

</div>

## About this notebook

Every hour of delay in treating sepsis worsens outcomes, which is why hospitals deploy early warning scores that watch vital signs and laboratory values around the clock. Such systems are validated with an area under the ROC curve, and a proprietary sepsis model in wide use performed far below its reported accuracy when examined externally [6]. This notebook shows a different gap: an attacker who changes vital signs by amounts smaller than routine measurement error can postpone the alert by hours, and no validation statistic registers anything.

**Data.** The PhysioNet/Computing in Cardiology Challenge 2019 released hourly records of intensive care patients from two hospital systems, labelled with the Sepsis-3 definition [3, 4, 5]. The notebook downloads a subset of training set A directly from PhysioNet's open data (CC BY 4.0). A septic patient's hours are labelled positive from six hours before the clinical onset time, so the task rewards early warning.

**Threat model.** The attacker sits in the data path between the bedside monitors and the early warning system, for example in a compromised interface engine or middleware. Each hour the attacker may shift the six charted vital signs within bounds that are plausible measurement noise: heart rate by 5 beats per minute, oxygen saturation by 1 percent, temperature by 0.2 °C, systolic and mean arterial pressure by 5 mmHg, and respiratory rate by 2 breaths per minute. Unlike the attacker in NB4, this one can read the risk score, for example from a dashboard, and uses it to estimate which changes lower the score: a zeroth-order attack in the spirit of ZOO [1]. Changes are made online, hour by hour, and once charted they remain part of the record that later hours see.

The full run takes four to six minutes on a free Colab CPU; most of it is the download of 2,500 patient files.

In [ ]:
import importlib.util, subprocess, sys
PACKAGES = {"shap": "shap>=0.46", "gemex": "gemex>=1.2.2"}
missing = [spec for module, spec in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
import os, io, sys, json, time, zipfile, hashlib, warnings, urllib.request, urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUICK = os.environ.get("XAI_WORKSHOP_QUICK", "0") == "1"   # smaller runs for automated checks
SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("workshop_data")
DATA_DIR.mkdir(exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
trapezoid = getattr(np, "trapezoid", None) or np.trapz

INK, ALERT, SAFE, MUTED, AMBER, VIOLET = "#1D2742", "#C0392B", "#1F7A6D", "#8A93A6", "#B26B00", "#6C4AB6"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "axes.titleweight": "bold", "axes.titlesize": 11,
})


def fetch(urls, dest, timeout=120):
    """Download the first reachable URL into dest, reuse an existing copy, and return the path."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in ([urls] if isinstance(urls, str) else urls):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                payload = response.read()
            part = dest.with_name(dest.name + ".part")
            part.write_bytes(payload)
            part.replace(dest)
            return dest
        except Exception as exc:
            errors.append(f"{url}: {exc}")
    raise RuntimeError("No source could be reached:\n" + "\n".join(errors))


print(f"numpy {np.__version__} | pandas {pd.__version__} | quick mode: {QUICK}")

In [ ]:
import shap
import ipywidgets as widgets
from concurrent.futures import ThreadPoolExecutor
from gemex import Explainer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

## 1. Data

In [ ]:
BASES = ["https://physionet-open.s3.amazonaws.com/challenge-2019/1.0.0/training/training_setA/",
         "https://physionet.org/files/challenge-2019/1.0.0/training/training_setA/"]
N_PATIENTS = 400 if QUICK else 2500
cache = DATA_DIR / f"sepsis_setA_{N_PATIENTS}.pkl"


def download_patient(pid):
    name = f"p{pid:06d}.psv"
    for base in BASES:
        try:
            with urllib.request.urlopen(base + name, timeout=60) as response:
                return pid, response.read()
        except Exception:
            continue
    return pid, None


if cache.exists():
    icu = pd.read_pickle(cache)
else:
    frames, next_id, t_start = [], 1, time.time()
    with ThreadPoolExecutor(max_workers=16) as pool:
        while len(frames) < N_PATIENTS and next_id < 25000:
            batch = range(next_id, next_id + 200)
            next_id += 200
            for pid, raw in pool.map(download_patient, batch):
                if raw is not None and len(frames) < N_PATIENTS:
                    frames.append(pd.read_csv(io.BytesIO(raw), sep="|").assign(patient=pid))
            if next_id > 1000 and not frames:
                raise RuntimeError("PhysioNet could not be reached. Check the internet connection and run the cell again.")
    icu = pd.concat(frames, ignore_index=True)
    icu.to_pickle(cache)
    print(f"downloaded {len(frames)} patient files in {time.time() - t_start:.0f} s")

septic = icu.groupby("patient")["SepsisLabel"].max()
print(f"{icu['patient'].nunique()} patients · {len(icu):,} patient-hours · "
      f"{septic.mean():.1%} develop sepsis · median stay {icu.groupby('patient').size().median():.0f} h")

## 2. An early warning model

Each hour is described by a state: the last six hours of the six vital signs (carried forward when a value was not charted), the latest values of five laboratory tests with flags for whether they have been measured yet, and four static variables. From the state the model computes 50 features: the current value, six-hour mean, minimum, maximum, spread and change of every vital sign, plus the laboratory and static data. A histogram gradient boosting model learns the label. Patients are split into training, validation and test groups, and the alert threshold is set on the validation group so that 5 percent of the hours of patients without sepsis trigger an alert, a common design target for alarm burden.

In [ ]:
VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "Resp"]
LABS = ["WBC", "Lactate", "Creatinine", "Platelets", "Bilirubin_total"]
STATIC = ["Age", "Gender", "ICULOS", "HospAdmTime"]
WIN, NV = 6, len(VITALS)
CUR = slice((WIN - 1) * NV, WIN * NV)             # position of the current hour inside a state
FEATURE_NAMES = ([f"{v}_{s}" for s in ["now", "mean6h", "min6h", "max6h", "sd6h", "change6h"] for v in VITALS]
                 + LABS + [f"{lab}_measured" for lab in LABS] + STATIC)


def patient_states(g):
    """One state per ICU hour, using only current and past data."""
    vit = g[VITALS].ffill().to_numpy(float)
    pad = np.vstack([np.full((WIN - 1, NV), np.nan), vit])
    windows = np.lib.stride_tricks.sliding_window_view(pad, (WIN, NV))[:, 0].reshape(len(g), WIN * NV)
    labs = g[LABS].ffill().to_numpy(float)
    return np.hstack([windows, labs, (~np.isnan(labs)).astype(float), g[STATIC].to_numpy(float)])


def state_features(S):
    S = np.atleast_2d(np.asarray(S, dtype=float))
    W = S[:, :WIN * NV].reshape(-1, WIN, NV)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        stats = [W[:, -1], np.nanmean(W, 1), np.nanmin(W, 1), np.nanmax(W, 1), np.nanstd(W, 1), W[:, -1] - W[:, 0]]
    return np.hstack(stats + [S[:, WIN * NV:]])


groups = {pid: g for pid, g in icu.groupby("patient", sort=False)}
S = np.vstack([patient_states(g) for g in groups.values()])
y = icu["SepsisLabel"].to_numpy(int)
P = icu["patient"].to_numpy()
hour = icu.groupby("patient", sort=False).cumcount().to_numpy()

patients = septic.index.to_numpy()
test_p = rng.choice(patients, size=int(0.3 * len(patients)), replace=False)
rest = np.setdiff1d(patients, test_p)
val_p = rng.choice(rest, size=int(0.2 * len(rest)), replace=False)
m_fit, m_val, m_te = ~np.isin(P, np.concatenate([test_p, val_p])), np.isin(P, val_p), np.isin(P, test_p)

clf = make_pipeline(FunctionTransformer(state_features),
                    SimpleImputer(strategy="median", keep_empty_features=True),
                    HistGradientBoostingClassifier(max_iter=100 if QUICK else 250, learning_rate=0.06,
                                                   max_leaf_nodes=31, l2_regularization=1.0,
                                                   class_weight="balanced", random_state=SEED))
clf.fit(S[m_fit], y[m_fit])


def risk(states):
    return clf.predict_proba(np.atleast_2d(states))[:, 1]


p_val = risk(S[m_val])
tau = float(np.quantile(p_val[septic.loc[P[m_val]].to_numpy() == 0], 0.95))
print(f"alert threshold: {tau:.3f}")

In [ ]:
test = pd.DataFrame({"patient": P[m_te], "hour": hour[m_te], "label": y[m_te], "risk": risk(S[m_te])})


def stay_summary(df):
    first_alert = df.loc[df["risk"] >= tau, "hour"].min()
    onset = df.loc[df["label"] == 1, "hour"].min() + 6 if df["label"].any() else np.nan
    return pd.Series({"septic": bool(df["label"].any()), "first_alert": first_alert,
                      "onset": min(onset, df["hour"].max()) if df["label"].any() else np.nan})


stays = test.groupby("patient")[["hour", "label", "risk"]].apply(stay_summary).astype({"septic": bool})
sep_stays = stays[stays["septic"]]
early = (sep_stays["first_alert"] <= sep_stays["onset"])
pd.Series({"hourly AUROC (test)": round(roc_auc_score(test["label"], test["risk"]), 3),
           "septic patients alerted before clinical onset": f"{early.mean():.0%}",
           "median warning time before onset (h)": (sep_stays["onset"] - sep_stays["first_alert"])[early].median(),
           "patients without sepsis with at least one alert": f"{stays.loc[~stays['septic'], 'first_alert'].notna().mean():.0%}"}
          ).to_frame("value")

## 3. The attack

For each septic test patient whose alert fired, the attacker replays the stay hour by hour. When the risk score reaches 90 percent of the threshold, the attacker probes each vital sign in both directions, estimates the local gradient from the score differences, and moves the values against it within the bounds. The margin of 10 percent keeps the score from drifting over the threshold with small fluctuations. Hours in which the score is already low are left untouched, which keeps the footprint small.

In [ ]:
BOUND = np.array([5.0, 1.0, 0.2, 5.0, 5.0, 2.0])                  # HR, O2Sat, Temp, SBP, MAP, Resp
PHYS_LO = np.array([20.0, 50.0, 30.0, 40.0, 20.0, 4.0])
PHYS_HI = np.array([250.0, 100.0, 43.0, 280.0, 200.0, 70.0])


def attack_stay(g, target, n_iter=6):
    """Streaming zeroth-order attack on one ICU stay."""
    states = patient_states(g)
    raw = g[VITALS].to_numpy(float)
    pad = np.vstack([np.full((WIN - 1, NV), np.nan), g[VITALS].ffill().to_numpy(float)])
    T = len(g)
    r_clean, r_att, forged, changed, queries = risk(states), np.empty(T), [], np.zeros(T, bool), 0
    for t in range(T):
        s = states[t].copy()
        s[:WIN * NV] = pad[t:t + WIN].ravel()
        cur = s[CUR].copy()
        free = ~np.isnan(cur)
        lo = np.where(free, np.maximum(cur - BOUND, PHYS_LO), cur)
        hi = np.where(free, np.minimum(cur + BOUND, PHYS_HI), cur)

        def score(V):
            batch = np.repeat(s[None], len(V), axis=0)
            batch[:, CUR] = V
            return risk(batch)

        x = cur.copy()
        r = score(x[None])[0]
        queries += 1
        for _ in range(n_iter):
            if r < target:
                break
            E = np.diag(np.where(free, BOUND / 2, 0.0))
            cand = np.vstack([np.clip(x + E, lo, hi), np.clip(x - E, lo, hi)])
            rc = score(cand)
            grad = (rc[:NV] - rc[NV:]) / BOUND
            step = np.clip(x - np.sign(np.nan_to_num(grad)) * BOUND / 2, lo, hi)
            rs_ = score(step[None])[0]
            queries += len(cand) + 1
            if rc.min() < rs_:
                x, r = cand[int(np.argmin(rc))], rc.min()
            else:
                x, r = step, rs_
        changed[t] = not np.allclose(x, cur, equal_nan=True)
        pad[t + WIN - 1] = x
        if t + 1 < T:                                   # carried-forward values inherit the manipulation
            carry = np.isnan(raw[t + 1])
            pad[t + WIN, carry] = x[carry]
        s[CUR] = x
        forged.append(s)
        r_att[t] = r
    return dict(r_clean=r_clean, r_att=r_att, forged=np.array(forged), changed=changed, queries=queries,
                vit_clean=g[VITALS].ffill().to_numpy(float),
                vit_att=pad[WIN - 1:].copy())


targets = sep_stays[sep_stays["first_alert"].notna()].index.to_numpy()
targets = rng.choice(targets, size=min(len(targets), 8 if QUICK else 40), replace=False)
runs, t_start = {}, time.time()
for pid in targets:
    runs[pid] = attack_stay(groups[pid], target=0.9 * tau, n_iter=4 if QUICK else 6)
print(f"{len(runs)} septic stays attacked in {time.time() - t_start:.0f} s")

rows = []
for pid, run in runs.items():
    clean_first = int(np.argmax(run["r_clean"] >= tau))
    fired = np.where(run["r_att"] >= tau)[0]
    rows.append({"patient": pid, "clean alert (h)": clean_first,
                 "alert under attack (h)": fired[0] if len(fired) else np.nan,
                 "stay length (h)": len(run["r_att"]), "onset (h)": sep_stays.loc[pid, "onset"],
                 "hours manipulated": int(run["changed"].sum()), "queries": run["queries"]})
res = pd.DataFrame(rows).set_index("patient")
suppressed = res["alert under attack (h)"].isna()
delay = (res["alert under attack (h)"] - res["clean alert (h)"])[~suppressed]
max_shift = np.nanmax(np.vstack([np.abs(r["vit_att"] - r["vit_clean"]) for r in runs.values()]), axis=0)
pd.Series({"stays attacked": len(res),
           "alert suppressed for the whole stay": f"{suppressed.mean():.0%}",
           "median delay when the alert still fired (h)": delay.median() if len(delay) else np.nan,
           "alerts no longer before clinical onset": int(((res["clean alert (h)"] <= res["onset (h)"])
                                                           & ~(res["alert under attack (h)"] <= res["onset (h)"])).sum()),
           "median queries per stay": int(res["queries"].median()),
           "largest change (HR, O2Sat, Temp, SBP, MAP, Resp)": ", ".join(f"{v:g}" for v in np.round(max_shift, 2))}
          ).to_frame("result")

In [ ]:
pid_show = res.assign(gain=res["alert under attack (h)"].fillna(res["stay length (h)"]) - res["clean alert (h)"]
                      ).sort_values("gain", ascending=False).index[0]


def plot_stay(pid):
    run = runs[pid]
    hrs = np.arange(len(run["r_att"]))
    fig, axes = plt.subplots(2, 2, figsize=(12, 5.6), sharex=True)
    ax = axes[0, 0]
    ax.plot(hrs, run["r_clean"], color=INK, lw=1.8, label="genuine record")
    ax.plot(hrs, run["r_att"], color=ALERT, lw=1.8, label="manipulated record")
    ax.axhline(tau, color=AMBER, ls="--", lw=1, label="alert threshold")
    if not np.isnan(sep_stays.loc[pid, "onset"]):
        ax.axvline(sep_stays.loc[pid, "onset"], color=VIOLET, ls=":", lw=1.5, label="clinical onset")
    ax.set_ylabel("sepsis risk")
    ax.legend(frameon=False, fontsize=7)
    for ax, j, unit in [(axes[0, 1], 0, "HR (bpm)"), (axes[1, 0], 4, "MAP (mmHg)"), (axes[1, 1], 5, "Resp (/min)")]:
        ax.plot(hrs, run["vit_clean"][:, j], color=INK, lw=1.4)
        ax.plot(hrs, run["vit_att"][:, j], color=ALERT, lw=1.4, alpha=0.85)
        ax.set_ylabel(unit)
    for ax in axes[1]:
        ax.set_xlabel("hours in the ICU")
    fig.suptitle(f"Patient {pid}: the same stay, genuine and manipulated", y=1.01)
    plt.tight_layout()
    plt.show()


plot_stay(pid_show)

The hourly AUROC reported above was computed on genuine records and does not move. From the perspective of every validation statistic the model is exactly as good as before; only the patients notice.

### The defender's toolkit

Three tools turn explanations into security signals. The first is an anomaly score on explanation vectors: a Mahalanobis distance measuring how unusual the SHAP attribution pattern of an input is, compared with genuine inputs that received the same decision. The second is a counterfactual margin: the smallest change in a clinically meaningful direction that would make the alert fire again. It is a counterfactual explanation in the sense of NB2, applied to a "no alert" decision, and it answers the question of how close that decision was to flipping. The third is the Page-Hinkley test [2], a sequential change detector that raises an alarm when the running mean of a monitored signal drifts upwards. Applied to a stream of scores, it looks for a sustained campaign rather than a single suspicious input.

In [ ]:
class MahalanobisDetector:
    """Distance of a vector from a reference population (mean and regularised covariance)."""

    def fit(self, Z):
        Z = np.asarray(Z, dtype=float)
        self.mu = Z.mean(axis=0)
        cov = np.cov(Z, rowvar=False) + 1e-6 * np.eye(Z.shape[1])
        self.precision = np.linalg.pinv(cov)
        return self

    def score(self, Z):
        D = np.asarray(Z, dtype=float) - self.mu
        return np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", D, self.precision, D), 0.0))


def page_hinkley(stream, delta, lam):
    """Page-Hinkley test for an upward shift in the mean. Returns (statistic trace, index of first alarm or None)."""
    mean, cum, cum_min, trace, alarm = 0.0, 0.0, 0.0, [], None
    for t, x in enumerate(stream, start=1):
        mean += (x - mean) / t
        cum += x - mean - delta
        cum_min = min(cum_min, cum)
        trace.append(cum - cum_min)
        if alarm is None and cum - cum_min > lam:
            alarm = t - 1
    return np.array(trace), alarm


def counterfactual_margin(decide, X, move, max_step=1.0, tol=0.01):
    """Smallest step s in [0, max_step] for which decide(move(X, s)) raises the alert; max_step if none does.

    move(X, s) applies a step of size s[i] to row i in a clinically meaningful direction.
    """
    X = np.atleast_2d(np.asarray(X, dtype=float))
    lo, hi = np.zeros(len(X)), np.full(len(X), float(max_step))
    active = np.asarray(decide(move(X, hi)), dtype=bool)
    while np.any(active & (hi - lo > tol)):
        mid = 0.5 * (lo + hi)
        fire = np.asarray(decide(move(X, mid)), dtype=bool)
        hi = np.where(active & fire, mid, hi)
        lo = np.where(active & ~fire, mid, lo)
    return np.where(active, hi, float(max_step))


def detection_auc(scores_clean, scores_attacked):
    """AUROC of a score that should be higher for manipulated inputs."""
    from sklearn.metrics import roc_auc_score
    s = np.concatenate([scores_clean, scores_attacked])
    lab = np.concatenate([np.zeros(len(scores_clean)), np.ones(len(scores_attacked))])
    return roc_auc_score(lab, s)


def campaign(clean_scores, attacked_scores, share=0.2, L=400, delta=0.25, seed=42):
    """Simulate a monitoring stream that switches halfway to a partly manipulated state and run Page-Hinkley.

    Scores are standardised with the median and interquartile range of genuine scores; the alarm threshold is
    set just above the largest statistic seen on 30 streams without any attack.
    """
    rs = np.random.default_rng(seed)
    clean_scores, attacked_scores = np.asarray(clean_scores, float), np.asarray(attacked_scores, float)
    med = np.median(clean_scores)
    iqr = np.subtract(*np.quantile(clean_scores, [0.75, 0.25])) + 1e-9
    z = lambda v: (np.asarray(v) - med) / iqr
    lam = 1.1 * max(page_hinkley(z(rs.choice(clean_scores, L)), delta, np.inf)[0].max() for _ in range(30))
    stream = rs.choice(clean_scores, L)
    t0 = L // 2
    hit = rs.random(L - t0) < share
    stream[t0:][hit] = rs.choice(attacked_scores, hit.sum())
    trace, alarm = page_hinkley(z(stream), delta, lam)
    return z(stream), trace, lam, alarm, t0


def plot_campaigns(signals, share=0.2):
    """signals: list of (title, genuine scores, manipulated scores); higher scores must mean more suspicious."""
    fig, axes = plt.subplots(1, len(signals), figsize=(6.5 * len(signals), 3.4), squeeze=False)
    for ax, (title, sc, sa) in zip(axes[0], signals):
        z_stream, trace_ph, lam, alarm, t0 = campaign(sc, sa, share=share)
        ax.scatter(np.arange(len(z_stream)), z_stream, s=5, alpha=0.5,
                   color=np.where(np.arange(len(z_stream)) >= t0, ALERT, INK))
        ax.axvline(t0, color=ALERT, ls="--", lw=1, label="campaign starts")
        if alarm is not None:
            ax.axvline(alarm, color=AMBER, lw=2, label=f"Page-Hinkley alarm after {alarm - t0} inputs")
        else:
            ax.text(0.02, 0.92, "no alarm", transform=ax.transAxes, color=AMBER, fontweight="bold")
        ax.set_title(title)
        ax.set_xlabel("near-threshold input in the stream")
        ax.set_ylabel("standardised score")
        ax.legend(frameon=False, fontsize=8, loc="lower left")
    plt.tight_layout()
    plt.show()

## 4. What explanations reveal

The analysis compares manipulated hours, those in which the attacker changed the record and the score stayed below the threshold, with genuine hours whose score lies in the same band just below the threshold. The SHAP anomaly score and the counterfactual margin are computed exactly as in NB4. For the margin, the direction is clinical deterioration: heart rate, temperature and respiratory rate rise, oxygen saturation and blood pressure fall, in steps measured in units of the attack bounds.

In [ ]:
S_att = np.vstack([run["forged"][run["changed"] & (run["r_att"] < tau)] for run in runs.values()])
band_idx = np.where(m_te)[0][(test["risk"] >= 0.5 * tau).to_numpy() & (test["risk"] < tau).to_numpy()
                              & (test["label"] == 0).to_numpy()]
band_idx = rng.permutation(band_idx)[:3000]
half = len(band_idx) // 2
features = clf[:-1]                                    # state -> imputed feature matrix
tree_explainer = shap.TreeExplainer(clf[-1])
S_ref_fit, S_ref_test = (tree_explainer.shap_values(features.transform(S[i])) for i in (band_idx[:half], band_idx[half:]))
S_att_shap = tree_explainer.shap_values(features.transform(S_att))

shap_det = MahalanobisDetector().fit(S_ref_fit)
s_clean, s_att = shap_det.score(S_ref_test), shap_det.score(S_att_shap)
raw_det = MahalanobisDetector().fit(features.transform(S[band_idx[:half]]))
print(f"{len(S_att)} manipulated hours · {len(band_idx) - half} genuine near-threshold hours for evaluation")
print(f"AUROC, SHAP anomaly score: {detection_auc(s_clean, s_att):.3f}")
print(f"AUROC, raw-feature distance: {detection_auc(raw_det.score(features.transform(S[band_idx[half:]])), raw_det.score(features.transform(S_att))):.3f}")

DIR = np.array([1.0, -1.0, 1.0, -1.0, -1.0, 1.0])


def worsen(states, s):
    out = np.array(states, dtype=float, copy=True)
    out[:, CUR] = np.clip(out[:, CUR] + np.asarray(s)[:, None] * DIR * BOUND, PHYS_LO, PHYS_HI)
    return out


def decide(states):
    return risk(states) >= tau


m_clean = counterfactual_margin(decide, S[band_idx[half:]], worsen, max_step=3.0)
m_att = counterfactual_margin(decide, S_att, worsen, max_step=3.0)
print(f"median counterfactual margin (in attack-bound units): genuine {np.median(m_clean):.2f} · "
      f"manipulated {np.median(m_att):.2f} · AUROC {detection_auc(-m_clean, -m_att):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
diff = pd.Series(S_att_shap.mean(axis=0) - S_ref_test.mean(axis=0), index=FEATURE_NAMES)
diff = diff.reindex(diff.abs().sort_values().index[-12:])
axes[0].barh(diff.index, diff.to_numpy(), color=[ALERT if v < 0 else SAFE for v in diff])
axes[0].set_xlabel("mean SHAP, manipulated minus genuine (log-odds)")
axes[0].set_title("Where the attack pushes the explanation")
bins = np.linspace(0, 3, 31)
axes[1].hist(m_clean, bins=bins, color=SAFE, alpha=0.7, density=True, label="genuine, near threshold")
axes[1].hist(m_att, bins=bins, color=ALERT, alpha=0.7, density=True, label="manipulated")
axes[1].set_xlabel("deterioration needed to trigger the alert (bound units)")
axes[1].legend(frameon=False)
plt.tight_layout()
plt.show()

**GEMEX on the attack surface.** The attacker controls six numbers per hour, so GEMEX [9] is applied to exactly those: the model is viewed as a function of the six current vital signs with everything else in the state held fixed. The geodesic length then measures how far, in the information geometry of the model, the charted vital signs lie from typical ones.

In [ ]:
class CurrentVitalsModel:
    """The alert model as a function of the six current vital signs, the rest of the state held fixed."""

    def __init__(self, state):
        self.state = np.asarray(state, dtype=float)

    def predict_proba(self, V):
        batch = np.repeat(self.state[None], len(np.atleast_2d(V)), axis=0)
        batch[:, CUR] = np.atleast_2d(V)
        p = risk(batch)
        return np.column_stack([1 - p, p])

    def predict(self, V):
        return (self.predict_proba(V)[:, 1] >= 0.5).astype(int)


complete = lambda A: A[~np.isnan(A[:, CUR]).any(axis=1)]
vit_ref = complete(S[np.where(m_fit)[0][:20000]])[:, CUR]
vit_ref = vit_ref[rng.choice(len(vit_ref), size=min(300, len(vit_ref)), replace=False)]


def geodesic_length(state):
    gx = Explainer(CurrentVitalsModel(state), data_type="tabular", feature_names=VITALS,
                   class_names=["no alert", "alert"])
    return gx.explain(state[CUR], X_reference=vit_ref).geodesic_lengths[-1]


gen_states, att_states = complete(S[band_idx[half:]]), complete(S_att)
n_g = min(4 if QUICK else 8, len(gen_states), len(att_states))
geo_clean = [geodesic_length(s) for s in gen_states[:n_g]]
geo_att = [geodesic_length(s) for s in att_states[:n_g]]
print(f"median geodesic length: genuine {np.median(geo_clean):.3f} · manipulated {np.median(geo_att):.3f} · "
      f"AUROC {detection_auc(geo_clean, geo_att):.3f} (n = {n_g} each)")

**Watching the unit.** A campaign against an intensive care unit shows up as a shift in the stream of near-threshold hours. The Page-Hinkley test runs on both signals, calibrated on streams without any attack.

In [ ]:
plot_campaigns([("SHAP anomaly score", s_clean, s_att), ("counterfactual margin (sign flipped)", -m_clean, -m_att)])

**What the evidence supports.** The score-based attacker needs many queries per hour, and each stays small, yet the alert can be delayed by hours or suppressed through the stay. The printed AUROC values show how far each explanation-based signal separates manipulated from genuine near-threshold hours. Because this attacker aims 10 percent below the threshold instead of stopping on the boundary, the counterfactual margin is less extreme than in NB4, which illustrates the trade-off an adaptive attacker faces. The defender's strongest position combines several signals and watches them over time.

## 5. Explore the attacked stays

In [ ]:
widgets.interact(plot_stay, pid=widgets.Dropdown(options=[int(p) for p in runs], value=int(pid_show), description="patient"));

## Discussion

1. The attack keeps every change within routine measurement noise. Which organisational control, rather than a model change, would make it harder?
2. The attacker in this notebook reads the risk score. Should an early warning system expose its score to users at all, or only the alert?
3. Which log records would an incident response team need to reconstruct this attack after the fact?

## Summary

A score-based attacker who changes charted vital signs within measurement noise can delay or suppress sepsis alerts while the model's validation performance remains untouched. Explanation-based signals, from attribution anomalies to counterfactual margins and the geometry of the model, give a defender a way to observe the manipulation, and a sequential test aggregates weak evidence into a campaign alarm. NB6 turns to a consumer wearable, where the attacker again sees only a yes or no.

## References

[1] P.-Y. Chen, H. Zhang, Y. Sharma, J. Yi, and C.-J. Hsieh, "ZOO: Zeroth order optimization based black-box attacks to deep neural networks without training substitute models," in *Proc. 10th ACM Workshop on Artificial Intelligence and Security (AISec)*, 2017, pp. 15–26. arXiv:1708.03999

[2] E. S. Page, "Continuous inspection schemes," *Biometrika*, vol. 41, no. 1/2, pp. 100–115, 1954. doi: 10.1093/biomet/41.1-2.100

[3] M. A. Reyna et al., "Early prediction of sepsis from clinical data: The PhysioNet/Computing in Cardiology Challenge 2019," *Critical Care Medicine*, vol. 48, no. 2, pp. 210–217, 2020. doi: 10.1097/CCM.0000000000004145

[4] M. A. Reyna, C. Josef, R. Jeter, S. P. Shashikumar, B. Moody, M. B. Westover, A. Sharma, S. Nemati, and G. D. Clifford, "Early Prediction of Sepsis from Clinical Data: The PhysioNet/Computing in Cardiology Challenge 2019" (version 1.0.0), PhysioNet, 2019. doi: 10.13026/v64v-d857

[5] M. Singer et al., "The third international consensus definitions for sepsis and septic shock (Sepsis-3)," *JAMA*, vol. 315, no. 8, pp. 801–810, 2016. doi: 10.1001/jama.2016.0287

[6] A. Wong et al., "External validation of a widely implemented proprietary sepsis prediction model in hospitalized patients," *JAMA Internal Medicine*, vol. 181, no. 8, pp. 1065–1070, 2021. doi: 10.1001/jamainternmed.2021.2626

[7] A. L. Goldberger et al., "PhysioBank, PhysioToolkit, and PhysioNet: Components of a new research resource for complex physiologic signals," *Circulation*, vol. 101, no. 23, pp. e215–e220, 2000. doi: 10.1161/01.CIR.101.23.e215

[8] S. M. Lundberg et al., "From local explanations to global understanding with explainable AI for trees," *Nature Machine Intelligence*, vol. 2, pp. 56–67, 2020. doi: 10.1038/s42256-019-0138-9

[9] U. Kose, "GEMEX: Geodesic Entropic Manifold Explainability," Python package, version 1.2.2, 2026. https://pypi.org/project/gemex/

[10] S. G. Finlayson, J. D. Bowers, J. Ito, J. L. Zittrain, A. L. Beam, and I. S. Kohane, "Adversarial attacks on medical machine learning," *Science*, vol. 363, no. 6433, pp. 1287–1289, 2019. doi: 10.1126/science.aaw4399